In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent 
sys.path.insert(0, str(project_root))

In [2]:
import json
import os
from pathlib import Path

from train_utils import build_train_val_tables, run_trial
from xgb_hypothesis_config import HYPOTHESES

In [3]:
project_root = Path().resolve().parent.parent  
sys.path.insert(0, str(project_root))

DATASET_ROOT = Path(
    os.getenv("EWS_DATASET_PATH", project_root / "data" / "EWS-Dataset")
)

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR   = DATASET_ROOT / "validation"
TEST_DIR  = DATASET_ROOT / "test"
X_train, y_train, val_img_paths, val_mask_paths = build_train_val_tables(TRAIN_DIR, VAL_DIR)

Train: 142 images
Val:   24 images
Building training table...
X_train: (1396180, 13)  y_train: (1396180,)


In [5]:
all_results = []
for trial in HYPOTHESES:
    print(f"Running: {trial['hypothesis']} ...")
    metrics = run_trial(
        trial["params"],
        X_train,
        y_train,
        val_img_paths,
        val_mask_paths,
        model_type="xgb", 
    )
    all_results.append({
        "hypothesis": trial["hypothesis"],
        "rationale":  trial["rationale"],
        "params":     trial["params"],
        **metrics,
    })
    print(
        f"  IoU={metrics['mean_iou']:.4f} ±{metrics['std_iou']:.4f} "
        f"train={metrics['train_time_s']:.1f}s"
    )

print("\nDone.")

Running: baseline ...
  IoU=0.8396 ±0.1298 train=4.3s
Running: shallower_trees ...
  IoU=0.8422 ±0.1282 train=3.4s
Running: lower_lr_more_trees ...
  IoU=0.8400 ±0.1295 train=9.6s
Running: more_row_sampling ...
  IoU=0.8399 ±0.1295 train=4.5s
Running: more_feature_sampling ...
  IoU=0.8404 ±0.1291 train=4.2s

Done.


In [ ]:
best = max(all_results, key=lambda x: x["mean_iou"])

output = {
    "best_hypothesis": best["hypothesis"],
    "best_params": best["params"],
    "best_iou": best["mean_iou"],
    "rationale": best["rationale"],
    "n_trials": len(all_results),
    "all_trials": all_results,
}

with open("xgb_hyperparam_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Best: {best['hypothesis']}  IoU={best['mean_iou']:.4f}")
print("Saved: xgb_hyperparam_results.json")

Best: shallower_trees  IoU=0.8421
Saved: xgb_hyperparam_results.json
